# Encoding Categorical Variables
### A first introduction, using the Titanic dataset

**Python for Data Science - Introduction to Data Preparation**

---

Most of the interesting information in a dataset arrives as **words**, not numbers. Male or female. First class or third. Southampton or Cherbourg.

Computers cannot do arithmetic on words. Almost every statistical method and every machine-learning model needs numbers. **Encoding** is the step where you turn the words into numbers.

It sounds mechanical. It is not. The way you convert a word to a number makes a *claim* about that word, and if the claim is false your results will be wrong in ways nothing warns you about.

This notebook covers the three encodings you will use constantly:

| Encoding | Use it when |
|---|---|
| **One-hot** | the categories have no order (male/female, ports, colours) |
| **Ordinal** | the categories have a real order (first/second/third class) |
| **Manual** | you know something about the data that the column does not say |

Everything is pandas. No modelling libraries required.

---

## 1. Setup

The Titanic dataset ships with seaborn, so there is nothing to download by hand. One row is one passenger - which makes this a good dataset to learn on, because there is never any confusion about what a row represents.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)

titanic = sns.load_dataset('titanic')

print(f'{titanic.shape[0]} passengers, {titanic.shape[1]} columns')
titanic.head()

891 passengers, 15 columns


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
titanic.dtypes.to_frame('dtype')

,dtype
survived,int64
pclass,int64
sex,object
age,float64
sibsp,int64
parch,int64
fare,float64
embarked,object
class,category
who,object


---

## 2. Start here: the same fact, wearing two different outfits

Before we encode anything, look at these six columns. They are really only **three pieces of information**, each stored twice - once as words, once as numbers.

In [3]:
titanic[['alive', 'survived',
         'class', 'pclass',
         'embark_town', 'embarked']].head(8)

,alive,survived,class,pclass,embark_town,embarked
0,no,0,Third,3,Southampton,S
1,yes,1,First,1,Cherbourg,C
2,yes,1,Third,3,Southampton,S
3,yes,1,First,1,Southampton,S
4,no,0,Third,3,Southampton,S
5,no,0,Third,3,Queenstown,Q
6,no,0,First,1,Southampton,S
7,no,0,Third,3,Southampton,S


Read across the pairs:

| Words | Numbers | The same fact? |
|---|---|---|
| `alive` = *no* | `survived` = *0* | yes |
| `class` = *Third* | `pclass` = *3* | yes |
| `embark_town` = *Southampton* | `embarked` = *S* | yes |

Nobody collected each fact twice. **Someone encoded them.** The right-hand column of each pair is the left-hand column after somebody did the work this notebook is about.

So encoding is not an abstract requirement imposed by a library. It is a thing you can already see the finished result of, right here, in three columns.

We can prove the first pair really is one fact:

In [4]:
manual_version = (titanic['alive'] == 'yes').astype(int)

print('does our own encoding of `alive` reproduce `survived` exactly?')
print('  ', manual_version.equals(titanic['survived']))
print()
print(pd.crosstab(titanic['alive'], titanic['survived']))

does our own encoding of `alive` reproduce `survived` exactly?
   True

survived    0    1
alive             
no        549    0
yes         0  342


The crosstab is perfectly diagonal - every *no* is a 0 and every *yes* is a 1. One fact, two columns.

---

## 3. Why it matters: pandas quietly ignores what it cannot read

Here is the question this dataset exists to answer: **what made a passenger more likely to survive?**

A correlation is the simplest way to start. Let us try one.

In [5]:
correlations = titanic.corr(numeric_only=True)['survived'].sort_values(ascending=False)

print('Correlation with survival:\n')
print(correlations.to_string())

Correlation with survival:

survived      1.000000
fare          0.257307
parch         0.081629
sibsp        -0.035322
age          -0.077221
alone        -0.203367
pclass       -0.338481
adult_male   -0.557080


That worked. But look at what is **missing** from the list.

In [6]:
used    = set(titanic.corr(numeric_only=True).columns)
ignored = [c for c in titanic.columns if c not in used]

print(f'columns in the dataset : {titanic.shape[1]}')
print(f'columns corr() used    : {len(used)}')
print(f'columns corr() ignored : {len(ignored)}')
print()
print('silently ignored:', ignored)

columns in the dataset : 15
columns corr() used    : 8
columns corr() ignored : 7

silently ignored: ['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']


**`sex` is not on the list.** On the Titanic, sex was far and away the strongest predictor of who lived - and our analysis dropped it without a word of warning.

That is the danger. Nothing crashed. No warning appeared. We got a clean, tidy, confident table of results that was missing the most important variable in the dataset.

> Notice we had to write `numeric_only=True` to make it work at all. That argument is pandas asking you to confirm you are happy to throw the text columns away. It is very easy to type it without thinking about what it discards.

We will come back to this exact correlation at the end of the notebook, once the text columns have been encoded, and see what changes.

---

## 4. The one distinction that matters: nominal vs ordinal

Before choosing an encoding, ask **one question** about the column:

> ### Do these categories have a natural order?

- **No order = nominal.** `sex`, `embarked`, `deck`. Southampton is not "more" than Cherbourg. There is no sensible way to line them up.
- **Real order = ordinal.** `class`. First beats second beats third. Everyone on board understood that ordering, and it mattered enormously.

Your answer decides the encoding:

| | Encoding | Why |
|---|---|---|
| **Nominal** | one-hot | it refuses to impose an order that does not exist |
| **Ordinal** | ordinal codes | it *preserves* an order that genuinely does exist |

Getting this backwards is the single most common encoding mistake, and we will look at it directly in section 7.

First, a quick inventory of what we are dealing with.

In [7]:
text_cols = titanic.select_dtypes(include=['object', 'category', 'bool']).columns

inventory = pd.DataFrame({
    'dtype'     : [str(titanic[c].dtype) for c in text_cols],
    'n_levels'  : [titanic[c].nunique() for c in text_cols],
    'n_missing' : [titanic[c].isna().sum() for c in text_cols],
    'values'    : [', '.join(map(str, sorted(titanic[c].dropna().unique())[:4]))
                   for c in text_cols],
}, index=text_cols)

inventory

,dtype,n_levels,n_missing,values
sex,object,2,0,"female, male"
embarked,object,3,2,"C, Q, S"
class,category,3,0,"First, Second, Third"
who,object,3,0,"child, man, woman"
adult_male,bool,2,0,"False, True"
deck,category,7,688,"A, B, C, D"
embark_town,object,3,2,"Cherbourg, Queenstown, Southampton"
alive,object,2,0,"no, yes"
alone,bool,2,0,"False, True"


Two things to notice, both of which we will deal with later:

- `embarked` and `embark_town` have a **small number of missing values**. Small enough to overlook, which is exactly why they cause trouble.
- `deck` is missing for most passengers. That is not a broken column - it turns out to be informative. Section 8.

---

## 5. One-hot encoding

One column per category, holding **1** if the row belongs to that category and **0** if it does not.

Start with the simplest possible case: `sex` has two values.

In [8]:
sex_dummies = pd.get_dummies(titanic['sex'], prefix='sex', dtype=int)

print('before:')
print(titanic['sex'].head(6).to_string())
print('\nafter:')
sex_dummies.head(6)

before:
0      male
1    female
2    female
3    female
4      male
5      male

after:


,sex_female,sex_male
0,0,1
1,1,0
2,1,0
3,1,0
4,0,1
5,0,1


That is the whole idea. `male` became `sex_male=1, sex_female=0`; `female` became the reverse.

> **Why `dtype=int`?** Without it, `get_dummies` gives you `True`/`False` instead of `1`/`0`. Both work, but `1`/`0` is easier to read and behaves more predictably when you start doing arithmetic.

Now a three-level column, `embarked` - the port where each passenger boarded.

In [9]:
embarked_dummies = pd.get_dummies(titanic['embarked'], prefix='port', dtype=int)

print('C = Cherbourg,  Q = Queenstown,  S = Southampton\n')
embarked_dummies.head(6)

C = Cherbourg,  Q = Queenstown,  S = Southampton



,port_C,port_Q,port_S
0,0,0,1
1,1,0,0
2,0,0,1
3,0,0,1
4,0,0,1
5,0,1,0


Three categories, three columns. Each row has exactly one `1`.

Except... let us check that claim, because it is not quite true.

In [10]:
row_totals = embarked_dummies.sum(axis=1)

print('how many 1s does each row have?\n')
print(row_totals.value_counts().to_string())

how many 1s does each row have?

1    889
0      2


### The missing-value trap

Two passengers have **no 1 at all** - a row of three zeros.

Those are the two passengers whose port was never recorded. `get_dummies` did not error, did not warn, and did not create a column for them. It just quietly encoded them as *not from anywhere*.

A row of all zeros does not mean "unknown". To every model you ever build, it means "definitely not Cherbourg, definitely not Queenstown, definitely not Southampton" - which is a statement about the passenger that is simply false.

Here they are:

In [11]:
missing_port = titanic['embarked'].isna()

print(f'passengers with no recorded port: {missing_port.sum()}\n')
titanic.loc[missing_port, ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'embark_town']]

passengers with no recorded port: 2



,survived,pclass,sex,age,fare,embarked,embark_town
61,1,1,female,38.0,80.0,NaN,NaN
829,1,1,female,62.0,80.0,NaN,NaN


In [12]:
# The fix: ask get_dummies to give missing values their own column.
embarked_fixed = pd.get_dummies(titanic['embarked'], prefix='port',
                                dtype=int, dummy_na=True)

print('columns now:', list(embarked_fixed.columns))
print()
print('rows with exactly one 1:', (embarked_fixed.sum(axis=1) == 1).all())
print()
embarked_fixed[embarked_fixed['port_nan'] == 1]

columns now: ['port_C', 'port_Q', 'port_S', 'port_nan']

rows with exactly one 1: True



,port_C,port_Q,port_S,port_nan
61,0,0,0,1
829,0,0,0,1


> **The habit to build:** after any `get_dummies`, check that every row sums to 1. It takes one line and it catches this immediately.
>
> ```python
> (dummies.sum(axis=1) == 1).all()
> ```

With only two affected passengers you might reasonably decide to drop them instead, or fill them with the most common port. All three choices are defensible. **Silently encoding them as zeros is not a choice at all** - it is what happens when you do not look.

### A note on `drop_first`

You will see this in tutorials:

In [13]:
print('both columns    :', list(pd.get_dummies(titanic['sex'], prefix='sex').columns))
print('with drop_first :', list(pd.get_dummies(titanic['sex'], prefix='sex',
                                               drop_first=True).columns))

both columns    : ['sex_female', 'sex_male']
with drop_first : ['sex_male']


`drop_first=True` throws away one column, because it is redundant: if you know `sex_male`, you already know `sex_female`. Whichever category gets dropped becomes the **baseline** that the others are compared against.

For now: **you usually do not need it.** It matters for linear and logistic regression, where keeping both columns causes a technical problem. For most other purposes - and for simply looking at your data, which is what we are doing - keeping both is clearer. Do not add it out of habit.

---

## 6. Ordinal encoding

Now the columns that *do* have an order. And Titanic gives us a small surprise here.

### `pclass` needs no work at all

In [14]:
print('pclass values :', sorted(titanic['pclass'].unique()))
print('pclass dtype  :', titanic['pclass'].dtype)
print()
print('It is already numeric, and the numbers already run in the right order.')
print('The correct encoding for this column is: do nothing.')

pclass values : [np.int64(1), np.int64(2), np.int64(3)]
pclass dtype  : int64

It is already numeric, and the numbers already run in the right order.
The correct encoding for this column is: do nothing.


This is worth pausing on, because beginners often assume every column needs transforming. It does not. `pclass` arrives as 1, 2, 3 - the ordering is already faithful, and 1st class really is "one better than" 2nd in the way the numbers imply.

### `class` holds the same information as words

Same facts, stored as text. And this is where it gets interesting.

In [15]:
print('dtype      :', titanic['class'].dtype)
print('categories :', list(titanic['class'].cat.categories))
print('ordered?   :', titanic['class'].cat.ordered)
print()

try:
    titanic['class'] < 'Third'
except TypeError as err:
    print('Comparing them raises TypeError:')
    print('  ', err)

dtype      : category
categories : ['First', 'Second', 'Third']
ordered?   : False

Comparing them raises TypeError:
   Unordered Categoricals can only compare equality or not


pandas stores `class` as a **categorical** column - but an *unordered* one. Even though the ordering is blindingly obvious to any human being, pandas refuses to guess it, so a comparison that should obviously work does not.

That refusal is a feature, not an annoyance. pandas will not invent an order for you, because inventing orders is precisely how encoding goes wrong. You have to state it.

In [16]:
CLASS_ORDER = ['First', 'Second', 'Third']          # stated explicitly, by us

class_ordered = pd.Categorical(titanic['class'],
                               categories=CLASS_ORDER,
                               ordered=True)

titanic['class_code'] = class_ordered.codes

print('mapping:')
for i, name in enumerate(CLASS_ORDER):
    print(f'  {name:<8} -> {i}')

print()
print('comparisons work now:')
print('  passengers in first class:', (class_ordered < 'Second').sum())

mapping:
  First    -> 0
  Second   -> 1
  Third    -> 2

comparisons work now:
  passengers in first class: 216


In [17]:
# Self-check: our codes should be pclass minus 1, since pclass runs 1-3 and codes run 0-2.
agree = (titanic['class_code'] == titanic['pclass'] - 1).all()

print('our ordinal encoding of `class` matches the existing `pclass` column:', agree)
print()
titanic[['class', 'class_code', 'pclass']].drop_duplicates().sort_values('pclass')

our ordinal encoding of `class` matches the existing `pclass` column: True



,class,class_code,pclass
1,First,0,1
9,Second,1,2
0,Third,2,3


We encoded `class` ourselves and got back a column that already existed. That is the best kind of exercise: the answer was in the dataset all along, so you can check your work.

### A lucky escape worth understanding

What if we had not bothered to state the order, and let pandas sort the categories itself?

In [18]:
lazy_codes = pd.Categorical(titanic['class']).codes      # pandas sorts alphabetically

print('alphabetical order :', sorted(CLASS_ORDER))
print('correct order      :', CLASS_ORDER)
print()
print('did the lazy version give the same codes?', bool((lazy_codes == class_ordered.codes).all()))

alphabetical order : ['First', 'Second', 'Third']
correct order      : ['First', 'Second', 'Third']

did the lazy version give the same codes? True


It worked. **By pure coincidence.**

"First", "Second", "Third" happen to be in alphabetical order as well as rank order. Change the words and the luck runs out immediately:

In [19]:
sizes = pd.Series(['Low', 'High', 'Medium', 'Low', 'High'])

print('alphabetical codes pandas would assign:')
for cat, code in zip(pd.Categorical(sizes).categories,
                     range(sizes.nunique())):
    print(f'  {cat:<8} -> {code}')

print('\nwhich claims High < Low < Medium. Nonsense.')
print('Always write the order out. Do not rely on getting lucky.')

alphabetical codes pandas would assign:
  High     -> 0
  Low      -> 1
  Medium   -> 2

which claims High < Low < Medium. Nonsense.
Always write the order out. Do not rely on getting lucky.


---

## 7. The classic mistake

Now the error you have been building towards: **using ordinal codes on a column that has no order.**

It is tempting because it is so little typing. One line, one tidy column of numbers, no explosion of new columns. Let us do it to `embarked` and see what we have actually claimed.

In [20]:
port_codes = pd.Categorical(titanic['embarked'])

print('the tempting one-liner:\n')
for cat, code in zip(port_codes.categories, range(len(port_codes.categories))):
    town = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}[cat]
    print(f'  {cat}  ({town:<12}) -> {code}')

the tempting one-liner:

  C  (Cherbourg   ) -> 0
  Q  (Queenstown  ) -> 1
  S  (Southampton ) -> 2


Those numbers look harmless. Read what they assert:

- Southampton (2) is **twice** Queenstown (1)
- Queenstown sits **exactly halfway between** Cherbourg and Southampton
- The gap from Cherbourg to Queenstown **equals** the gap from Queenstown to Southampton

Every one of those statements is meaningless. They are three towns on a map, not points on a scale.

And here is the proof that the numbers carry no information - change nothing but the order we list the categories in, and every passenger's code changes:

In [21]:
alphabetical = pd.Categorical(titanic['embarked'])
by_size      = pd.Categorical(titanic['embarked'],
                              categories=titanic['embarked'].value_counts().index)

comparison = pd.DataFrame({
    'port'               : titanic['embarked'],
    'code (alphabetical)': alphabetical.codes,
    'code (by size)'     : by_size.codes,
}).dropna().drop_duplicates().sort_values('port')

print('Two equally reasonable orderings, two completely different sets of numbers:\n')
comparison

Two equally reasonable orderings, two completely different sets of numbers:



,port,code (alphabetical),code (by size)
1,C,0,1
5,Q,1,2
0,S,2,0


Both encodings are "valid". They disagree. So any conclusion that depends on those numbers is a conclusion about **which ordering you happened to pick**, not about the Titanic.

That is the test to remember:

> **If reshuffling the categories changes your answer, the numbers were never meaningful.** Use one-hot instead.

---

## 8. Manual encoding

Sometimes you know something about the data that no single column states. Writing that knowledge down as a new column is **manual encoding**, and it is often the most valuable step in the whole process.

### Rebuilding a column that already exists

The dataset has a `who` column with values *man*, *woman*, *child*. That column is not raw data - somebody derived it from `sex` and `age`. Let us work out the rule and rebuild it ourselves, then check.

In [22]:
print(titanic['who'].value_counts().to_string())
print()
print('oldest passenger labelled "child":',
      titanic.loc[titanic['who'] == 'child', 'age'].max())
print('youngest passenger labelled "man" or "woman":',
      titanic.loc[titanic['who'] != 'child', 'age'].min())

who
man      537
woman    271
child     83

oldest passenger labelled "child": 15.0
youngest passenger labelled "man" or "woman": 16.0


In [23]:
# The rule looks like: under 16 -> child, otherwise split by sex.
def classify(row):
    if pd.notna(row['age']) and row['age'] < 16:
        return 'child'
    return 'man' if row['sex'] == 'male' else 'woman'

my_who = titanic.apply(classify, axis=1)

match_rate = (my_who == titanic['who']).mean()
print(f'our rule reproduces the existing `who` column for {match_rate:.1%} of passengers')

# NOTE: build the comparison column FIRST, then filter. Filtering first and
# calling .assign() afterwards silently re-expands an empty result back to
# full length - a nasty little pandas gotcha.
comparison = titanic.assign(our_rule=my_who)
disagreements = comparison.loc[my_who != titanic['who'], ['sex', 'age', 'who', 'our_rule']]

print(f'rows where we disagree: {len(disagreements)}')
if len(disagreements):
    display(disagreements.head(10))
else:
    print('Our rule matched the original exactly. Try changing 16 to 18 and re-running')
    print('to see what disagreement looks like.')

our rule reproduces the existing `who` column for 100.0% of passengers
rows where we disagree: 0
Our rule matched the original exactly. Try changing 16 to 18 and re-running
to see what disagreement looks like.


If your rule does not match perfectly, **that is the interesting part**, not a failure. Look at the rows where you disagree and work out whether the original author used a different age cutoff, or handled missing ages differently. Reverse-engineering someone else's encoding teaches you more than inventing one from scratch.

### Turning "missing" into information

`deck` is missing for most passengers. The instinct is to throw the column away. Look first.

In [24]:
print(f'`deck` is missing for {titanic["deck"].isna().mean():.0%} of passengers\n')

titanic['has_cabin'] = titanic['deck'].notna().astype(int)

summary = titanic.groupby('has_cabin').agg(
    passengers     = ('survived', 'size'),
    survival_rate  = ('survived', 'mean'),
    mean_fare      = ('fare', 'mean'),
    pct_first_class= ('pclass', lambda s: (s == 1).mean()),
)
summary.index = ['cabin NOT recorded', 'cabin recorded']
summary.style.format({'survival_rate': '{:.1%}', 'mean_fare': '{:,.1f}',
                      'pct_first_class': '{:.1%}'})

`deck` is missing for 77% of passengers



,passengers,survival_rate,mean_fare,pct_first_class
cabin NOT recorded,688,29.9%,19.2,6.0%
cabin recorded,203,67.0%,76.3,86.2%


The passengers with a recorded cabin survived at a dramatically higher rate - because having a recorded cabin is largely a proxy for travelling first class.

So `deck` being missing is **not random**, and it is **not noise**. The absence of the value is itself a fact about the passenger, and one line of manual encoding captured it.

> This is a general lesson worth carrying: before deleting a column for having too many missing values, check whether the *pattern* of missingness predicts anything.

---

## 9. What not to encode

One short warning before we assemble everything.

Remember `alive` from section 2 - the column that is identical to `survived`. If you are trying to predict survival and you encode `alive` as an input, you have handed the model the answer.

In [25]:
print('is `alive` just `survived` in words?',
      (titanic['alive'] == 'yes').astype(int).equals(titanic['survived']))
print()
print('So `alive` must never be used as an input to predict `survived`.')
print('It would give a perfect result that means absolutely nothing.')

is `alive` just `survived` in words? True

So `alive` must never be used as an input to predict `survived`.
It would give a perfect result that means absolutely nothing.


This is called **leakage**, and on real projects it is depressingly common - usually in subtler forms than this. The habit to build is to ask of every column: *would I actually know this value at the moment I need to make the prediction?*

Also skip: `adult_male` and `alone` are already `True`/`False`, so they are already numeric and need nothing. And `embark_town` is just `embarked` spelled out - encode one or the other, not both.

---

## 10. Putting it together

Every decision from this notebook, applied at once.

In [26]:
model_ready = pd.concat([
    # already numeric - nothing to do
    titanic[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']],

    # already boolean - just make them 0/1
    titanic[['adult_male', 'alone']].astype(int),

    # nominal -> one-hot
    pd.get_dummies(titanic['sex'], prefix='sex', dtype=int),
    pd.get_dummies(titanic['embarked'], prefix='port', dtype=int, dummy_na=True),

    # manual encoding from section 8
    titanic[['has_cabin']],
], axis=1)

print(f'original  : {titanic.shape[0]} rows x {titanic.shape[1]} columns')
print(f'encoded   : {model_ready.shape[0]} rows x {model_ready.shape[1]} columns')
print()
print('every column is now numeric:', model_ready.dtypes.map(pd.api.types.is_numeric_dtype).all())
model_ready.head()

original  : 891 rows x 17 columns
encoded   : 891 rows x 15 columns

every column is now numeric: True


,survived,pclass,age,sibsp,parch,fare,adult_male,alone,sex_female,sex_male,port_C,port_Q,port_S,port_nan,has_cabin
0,0,3,22.0,1,0,7.2500,1,0,0,1,0,0,1,0,0
1,1,1,38.0,1,0,71.2833,0,0,1,0,1,0,0,0,1
2,1,3,26.0,0,0,7.9250,0,1,1,0,0,0,1,0,0
3,1,1,35.0,1,0,53.1000,0,0,1,0,0,0,1,0,1
4,0,3,35.0,0,0,8.0500,1,1,0,1,0,0,1,0,0


> **On what we did and did not include.** `class_code` is left out because `pclass` already carries exactly the same information - encoding one fact twice adds nothing.
>
> `adult_male` *is* kept alongside `sex_male`, because despite appearances they are not the same column: a 10-year-old boy is `sex_male=1` but `adult_male=0`. It combines sex and age into one flag, which is itself a small piece of manual encoding somebody did for us.

### Back to the question we could not answer

Section 3 tried to correlate everything with survival, and pandas silently dropped `sex`. Let us run exactly the same analysis on the encoded data.

In [27]:
encoded_corr = model_ready.corr()['survived'].drop('survived').sort_values(ascending=False)

print('Correlation with survival, now that every column is readable:\n')
print(encoded_corr.to_string())

Correlation with survival, now that every column is readable:

sex_female    0.543351
has_cabin     0.319572
fare          0.257307
port_C        0.168240
parch         0.081629
port_nan      0.060095
port_Q        0.003650
sibsp        -0.035322
age          -0.077221
port_S       -0.155660
alone        -0.203367
pclass       -0.338481
sex_male     -0.543351
adult_male   -0.557080


**`sex_female` is at the very top, and `sex_male` at the very bottom.** Being female was by a distance the strongest single predictor of surviving the Titanic - stronger than class, stronger than fare, stronger than age.

In section 3 that finding was invisible. The data had not changed at all. The only thing that changed is that we encoded the column so pandas could see it.

> This is the entire argument for taking encoding seriously. It is not a formatting chore you do to satisfy a library. **An unencoded column is an invisible column**, and the thing you most need to know may be hiding in it.

---

## 11. Summary

**Ask one question first:** do the categories have a real order?

| Column looks like | Do this | pandas |
|---|---|---|
| No natural order (`sex`, `embarked`) | one-hot | `pd.get_dummies(col, dtype=int)` |
| Real order (`class`) | ordinal codes, order stated | `pd.Categorical(col, categories=[...], ordered=True).codes` |
| Already numeric and correctly ordered (`pclass`) | **nothing** | - |
| Already `True`/`False` (`alone`) | just cast | `col.astype(int)` |
| Knowledge not in any column | manual encoding | `.apply()`, `.map()`, `notna()` |
| Duplicate of another column (`embark_town`) | pick one | - |
| Gives away the answer (`alive`) | **exclude** | - |

**Five things to remember**

1. An unencoded column is invisible - `numeric_only=True` throws it away without warning.
2. Nominal gets one-hot. Ordinal gets codes. Mixing them up invents facts.
3. If reshuffling the categories changes your answer, the numbers were meaningless.
4. After `get_dummies`, check `(dummies.sum(axis=1) == 1).all()`. Missing values become all-zero rows.
5. Write orderings out explicitly. Alphabetical order is right only by luck.

---
